In [ ]:

# ============================================================
# GENERATIVE AI LABORATORY
# EXPERIMENT 8: SEMANTIC SEARCH USING EMBEDDINGS AND FAISS
# ============================================================

!pip install -q sentence-transformers faiss-cpu pandas numpy

import re
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from IPython.display import display

# ------------------------------------------------------------
# 1. LOAD DOCUMENTS
# ------------------------------------------------------------

documents = [
    {
        "doc_id": "D1",
        "title": "Introduction to Generative AI",
        "source": "Student-created demonstration text",
        "text": """
        Generative Artificial Intelligence is a branch of AI that creates
        new content such as text, images, audio, video, and computer code.
        It learns patterns from training data and uses those patterns to
        generate new outputs. Large language models are an important
        example of generative AI because they can answer questions,
        summarize information, translate languages, and assist with writing.
        Generative AI systems may use neural networks such as transformers.
        These systems do not simply copy training examples in every case.
        They generate responses based on learned statistical patterns.
        Applications include education, healthcare, software development,
        research, and customer support. However, generated answers can
        contain incorrect information, bias, or fabricated details.
        Therefore, their outputs should be checked for accuracy, privacy,
        fairness, and reliability before being used in important decisions.
        """
    },
    {
        "doc_id": "D2",
        "title": "Natural Language Processing",
        "source": "Student-created demonstration text",
        "text": """
        Natural Language Processing, commonly called NLP, is a field of
        artificial intelligence that helps computers process human language.
        NLP systems work with written text and sometimes spoken language.
        Common NLP tasks include sentiment analysis, text classification,
        machine translation, summarization, question answering, and named
        entity recognition. Text preprocessing may include removing
        unnecessary whitespace, tokenization, and normalization.
        Tokenization divides text into smaller units such as words or
        subwords. Traditional NLP systems often rely on keywords and
        manually designed features. Modern systems use neural networks
        and contextual representations to understand relationships between
        words. The meaning of a word may change depending on its context.
        NLP is used in search engines, virtual assistants, chatbots,
        document analysis, and language-learning applications.
        """
    },
    {
        "doc_id": "D3",
        "title": "Attention Mechanism in Transformers",
        "source": "Student-created demonstration text",
        "text": """
        The attention mechanism allows a neural network to focus on
        important parts of an input when processing information.
        In Natural Language Processing, attention helps a model determine
        which words are relevant to other words in a sentence.
        Self-attention compares tokens with one another to calculate
        how much attention each token should receive. It uses query,
        key, and value representations to calculate attention weights
        and combine information from different tokens.
        Transformers use attention to process relationships between
        words and capture contextual information. For example, when
        interpreting a pronoun, a model may attend to a noun mentioned
        earlier in the sentence. Multi-head attention performs several
        attention operations in parallel so the model can capture
        different relationships. Attention is an important component
        of many modern language models and transformer architectures.
        """
    },
    {
        "doc_id": "D4",
        "title": "Text Embeddings and Vector Search",
        "source": "Student-created demonstration text",
        "text": """
        Text embeddings are numerical vectors that represent the meaning
        of text. An embedding model converts a sentence, paragraph, or
        document into a fixed-dimensional vector. Texts with similar
        meanings may have nearby vectors in the embedding space.
        Semantic search uses these vectors to retrieve information based
        on meaning rather than relying only on exact keyword matches.
        A document collection is split into chunks and each chunk is
        converted into an embedding. When a user submits a query, the
        same embedding model converts the query into a vector.
        The query vector is compared with document vectors using a
        similarity metric such as cosine similarity. A vector index
        such as FAISS can efficiently find the nearest vectors.
        Retrieved chunks can then be ranked and displayed with their
        document IDs, text, and metadata. Good chunking and consistent
        preprocessing can improve retrieval quality.
        """
    },
    {
        "doc_id": "D5",
        "title": "Retrieval-Augmented Generation",
        "source": "Student-created demonstration text",
        "text": """
        Retrieval-Augmented Generation, known as RAG, combines information
        retrieval with a generative language model. Instead of relying
        only on information stored in model parameters, a RAG system
        retrieves relevant passages from an external document collection.
        First, documents are cleaned, divided into chunks, and represented
        by embeddings. These embeddings are stored in a vector database
        or search index. When a user asks a question, the question is
        embedded and similar document chunks are retrieved.
        The retrieved passages are added to the prompt as context.
        A language model then uses the question and retrieved context
        to generate an answer. RAG can help ground responses in supplied
        documents and make private or updated information searchable.
        Its performance depends on document quality, chunking, retrieval
        accuracy, and the language model's ability to use the context.
        RAG does not guarantee that every generated answer is correct.
        """
    }
]

print("DOCUMENT LOADING SUCCESSFUL")
print("Total documents:", len(documents))

document_table = pd.DataFrame([
    {
        "Document ID": doc["doc_id"],
        "Title": doc["title"],
        "Source": doc["source"]
    }
    for doc in documents
])

display(document_table)


# ------------------------------------------------------------
# 2. CLEAN AND SPLIT DOCUMENTS INTO CHUNKS
# ------------------------------------------------------------

def clean_text(text):
    """Remove unnecessary whitespace."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_chunks(text, chunk_size=50, overlap=10):
    """Split text into overlapping word-based chunks."""
    words = clean_text(text).split()

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "Overlap must be non-negative and smaller than chunk size."
        )

    chunks_list = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]

        if not chunk_words:
            break

        chunks_list.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks_list


chunks = []

# FIXED INDENTATION: Both loops are correctly nested.
for doc in documents:
    document_chunks = split_into_chunks(
        doc["text"],
        chunk_size=50,
        overlap=10
    )

    for chunk_number, chunk_text in enumerate(
        document_chunks, start=1
    ):
        chunks.append({
            "chunk_id": f'{doc["doc_id"]}-C{chunk_number}',
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "source": doc["source"],
            "text": chunk_text
        })

chunks_df = pd.DataFrame(chunks)

print("\nCHUNKING COMPLETED")
print("Total document chunks:", len(chunks))

if len(chunks) < 15:
    raise ValueError(
        f"Only {len(chunks)} chunks were created. "
        "Add more document text or reduce chunk_size."
    )

display(
    chunks_df[["chunk_id", "doc_id", "title", "source"]]
)

print("\nSAMPLE CHUNK:")
print("Chunk ID:", chunks[0]["chunk_id"])
print("Text:", chunks[0]["text"])


# ------------------------------------------------------------
# 3. LOAD THE EMBEDDING MODEL
# ------------------------------------------------------------

model_name = "sentence-transformers/all-MiniLM-L6-v2"

print("\nLoading embedding model:", model_name)

model = SentenceTransformer(model_name)

print("Embedding model loaded successfully.")


# ------------------------------------------------------------
# 4. GENERATE DOCUMENT EMBEDDINGS
# ------------------------------------------------------------

chunk_texts = [chunk["text"] for chunk in chunks]

document_embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

embedding_dimension = document_embeddings.shape[1]

print("\nDOCUMENT EMBEDDINGS GENERATED")
print("Number of vectors:", document_embeddings.shape[0])
print("Embedding dimension:", embedding_dimension)
print("Embedding array shape:", document_embeddings.shape)
print("Embedding model:", model_name)

assert len(document_embeddings) == len(chunks)
assert embedding_dimension == 384


# ------------------------------------------------------------
# 5. CREATE FAISS VECTOR INDEX
# ------------------------------------------------------------

# Inner product on normalized vectors equals cosine similarity.
index = faiss.IndexFlatIP(embedding_dimension)
index.add(document_embeddings)

print("\nFAISS INDEX CREATED SUCCESSFULLY")
print("Index type: IndexFlatIP")
print("Similarity metric: Cosine similarity")
print("Number of indexed vectors:", index.ntotal)

assert index.ntotal == len(chunks)


# ------------------------------------------------------------
# 6. DEFINE FIVE TEST QUERIES
# ------------------------------------------------------------

queries = [
    {
        "query_id": "Q1",
        "query": "What is attention in NLP?",
        "type": "Close wording",
        "expected_topic": "Attention Mechanism in Transformers"
    },
    {
        "query_id": "Q2",
        "query": "What are text embeddings and how are they used for semantic search?",
        "type": "Close wording",
        "expected_topic": "Text Embeddings and Vector Search"
    },
    {
        "query_id": "Q3",
        "query": "How does a model focus on important words in a sentence?",
        "type": "Paraphrased query",
        "expected_topic": "Attention Mechanism in Transformers"
    },
    {
        "query_id": "Q4",
        "query": "How can a system find passages with meanings similar to a question?",
        "type": "Paraphrased query",
        "expected_topic": "Text Embeddings and Vector Search"
    },
    {
        "query_id": "Q5",
        "query": "How do quantum computers implement quantum entanglement?",
        "type": "Information absent from corpus",
        "expected_topic": None
    }
]

print("\nTEST QUERIES")
display(pd.DataFrame(queries))


# ------------------------------------------------------------
# 7. SEMANTIC SEARCH FUNCTION
# ------------------------------------------------------------

def semantic_search(query, top_k=3):
    """Retrieve the top-k most similar document chunks."""

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores, positions = index.search(query_embedding, top_k)

    results = []

    for rank, (score, position) in enumerate(
        zip(scores[0], positions[0]), start=1
    ):
        if position < 0:
            continue

        chunk = chunks[int(position)]

        results.append({
            "Rank": rank,
            "Chunk ID": chunk["chunk_id"],
            "Document ID": chunk["doc_id"],
            "Document Title": chunk["title"],
            "Similarity": round(float(score), 4),
            "Retrieved Text": chunk["text"]
        })

    return results


# ------------------------------------------------------------
# 8. RUN ALL QUERIES AND DISPLAY RETRIEVED TEXT
# ------------------------------------------------------------

all_results = []

for item in queries:
    query_id = item["query_id"]
    query = item["query"]

    print("\n" + "=" * 80)
    print(f'{query_id}: {query}')
    print("Query type:", item["type"])
    print("Top-k: 3")
    print("=" * 80)

    results = semantic_search(query, top_k=3)

    for result in results:
        result["Query ID"] = query_id
        result["Query"] = query
        result["Query Type"] = item["type"]
        result["Expected Topic"] = item["expected_topic"]

        all_results.append(result)

        print("\nRank:", result["Rank"])
        print("Chunk ID:", result["Chunk ID"])
        print("Document:", result["Document Title"])
        print("Cosine similarity:", result["Similarity"])
        print("Retrieved text:")
        print(result["Retrieved Text"])


# ------------------------------------------------------------
# 9. RELEVANCE EVALUATION
# ------------------------------------------------------------

def evaluate_relevance(result):
    """
    Initial topic-based relevance judgement.
    Manually verify the returned text before submission.
    """
    expected_topic = result["Expected Topic"]
    actual_title = result["Document Title"]

    if expected_topic is None:
        return (
            "Irrelevant",
            "The corpus contains no document about quantum computing "
            "or quantum entanglement."
        )

    if actual_title == expected_topic:
        return (
            "Relevant",
            "The retrieved passage addresses the intended topic."
        )

    related_titles = {
        "Attention Mechanism in Transformers": {
            "Natural Language Processing",
            "Introduction to Generative AI"
        },
        "Text Embeddings and Vector Search": {
            "Retrieval-Augmented Generation",
            "Natural Language Processing"
        }
    }

    if actual_title in related_titles.get(expected_topic, set()):
        return (
            "Partially Relevant",
            "The passage provides related background but does not "
            "directly address the main topic."
        )

    return (
        "Irrelevant",
        "The passage does not directly address the intended topic."
    )


for result in all_results:
    relevance, reason = evaluate_relevance(result)
    result["Relevance"] = relevance
    result["Reason"] = reason


# ------------------------------------------------------------
# 10. DISPLAY COMPLETE RESULTS TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(all_results)

print("\nCOMPLETE RETRIEVAL RESULTS AND RELEVANCE EVALUATION")

display(
    results_df[
        [
            "Query ID",
            "Query",
            "Rank",
            "Chunk ID",
            "Document Title",
            "Similarity",
            "Retrieved Text",
            "Relevance",
            "Reason"
        ]
    ]
)


# ------------------------------------------------------------
# 11. RELEVANCE SUMMARY
# ------------------------------------------------------------

print("\nRELEVANCE SUMMARY")

relevance_summary = (
    results_df.groupby(["Query ID", "Relevance"])
    .size()
    .unstack(fill_value=0)
)

display(relevance_summary)

print("\nQUERY-WISE TOP RESULT")

top_results = results_df[
    results_df["Rank"] == 1
][
    [
        "Query ID",
        "Query",
        "Chunk ID",
        "Document Title",
        "Similarity",
        "Relevance"
    ]
]

display(top_results)


# ------------------------------------------------------------
# 12. PARAPHRASED QUERY TEST
# ------------------------------------------------------------

print("\nPARAPHRASE TEST")

for query_id in ["Q1", "Q3", "Q2", "Q4"]:
    subset = results_df[
        (results_df["Query ID"] == query_id) &
        (results_df["Rank"] == 1)
    ]

    if not subset.empty:
        row = subset.iloc[0]

        print("\nQuery ID:", row["Query ID"])
        print("Query:", row["Query"])
        print("Top result:", row["Document Title"])
        print("Chunk ID:", row["Chunk ID"])
        print("Similarity:", row["Similarity"])
        print("Relevance:", row["Relevance"])

print(
    "\nCompare Q1 with Q3 and Q2 with Q4. "
    "These pairs express related meanings using different wording. "
    "Check their retrieved text to evaluate semantic retrieval."
)


# ------------------------------------------------------------
# 13. OUT-OF-DOMAIN QUERY TEST
# ------------------------------------------------------------

print("\nOUT-OF-DOMAIN QUERY TEST")

out_of_domain = results_df[
    results_df["Query ID"] == "Q5"
]

display(
    out_of_domain[
        [
            "Query",
            "Rank",
            "Chunk ID",
            "Document Title",
            "Similarity",
            "Relevance",
            "Reason"
        ]
    ]
)

print(
    "FAISS returns the nearest available vectors even if the corpus "
    "does not contain the answer. Similarity alone does not prove "
    "that a retrieved passage answers the question."
)


# ------------------------------------------------------------
# 14. FINAL EXPERIMENT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPERIMENT 8: EXECUTION SUMMARY")
print("=" * 65)

print("Documents loaded:", len(documents))
print("Chunks created:", len(chunks))
print("Embedding model:", model_name)
print("Embedding dimension:", embedding_dimension)
print("FAISS index type: IndexFlatIP")
print("Similarity metric: Cosine similarity")
print("Vectors indexed:", index.ntotal)
print("Test queries executed:", len(queries))
print("Top-k per query: 3")
print("Total retrieved results:", len(results_df))

print("\nExperiment execution completed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 3.3 MB/s eta 0:00:00
DOCUMENT LOADING SUCCESSFUL
Total documents: 5


,Document ID,Title,Source
0,D1,Introduction to Generative AI,Student-created demonstration text
1,D2,Natural Language Processing,Student-created demonstration text
2,D3,Attention Mechanism in Transformers,Student-created demonstration text
3,D4,Text Embeddings and Vector Search,Student-created demonstration text
4,D5,Retrieval-Augmented Generation,Student-created demonstration text



CHUNKING COMPLETED
Total document chunks: 17


,chunk_id,doc_id,title,source
0,D1-C1,D1,Introduction to Generative AI,Student-created demonstration text
1,D1-C2,D1,Introduction to Generative AI,Student-created demonstration text
2,D1-C3,D1,Introduction to Generative AI,Student-created demonstration text
3,D2-C1,D2,Natural Language Processing,Student-created demonstration text
4,D2-C2,D2,Natural Language Processing,Student-created demonstration text
5,D2-C3,D2,Natural Language Processing,Student-created demonstration text
6,D3-C1,D3,Attention Mechanism in Transformers,Student-created demonstration text
7,D3-C2,D3,Attention Mechanism in Transformers,Student-created demonstration text
8,D3-C3,D3,Attention Mechanism in Transformers,Student-created demonstration text
9,D4-C1,D4,Text Embeddings and Vector Search,Student-created demonstration text



SAMPLE CHUNK:
Chunk ID: D1-C1
Text: Generative Artificial Intelligence is a branch of AI that creates new content such as text, images, audio, video, and computer code. It learns patterns from training data and uses those patterns to generate new outputs. Large language models are an important example of generative AI because they can answer questions,

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


... (22 outputs left out)

,Query ID,Query,Chunk ID,Document Title,Similarity,Relevance
0,Q1,What is attention in NLP?,D3-C1,Attention Mechanism in Transformers,0.6974,Relevant
3,Q2,What are text embeddings and how are they used...,D4-C1,Text Embeddings and Vector Search,0.8847,Relevant
6,Q3,How does a model focus on important words in a...,D3-C3,Attention Mechanism in Transformers,0.5685,Relevant
9,Q4,How can a system find passages with meanings s...,D5-C2,Retrieval-Augmented Generation,0.6353,Partially Relevant
12,Q5,How do quantum computers implement quantum ent...,D3-C1,Attention Mechanism in Transformers,0.1941,Irrelevant



PARAPHRASE TEST

Query ID: Q1
Query: What is attention in NLP?
Top result: Attention Mechanism in Transformers
Chunk ID: D3-C1
Similarity: 0.6974
Relevance: Relevant

Query ID: Q3
Query: How does a model focus on important words in a sentence?
Top result: Attention Mechanism in Transformers
Chunk ID: D3-C3
Similarity: 0.5685
Relevance: Relevant

Query ID: Q2
Query: What are text embeddings and how are they used for semantic search?
Top result: Text Embeddings and Vector Search
Chunk ID: D4-C1
Similarity: 0.8847
Relevance: Relevant

Query ID: Q4
Query: How can a system find passages with meanings similar to a question?
Top result: Retrieval-Augmented Generation
Chunk ID: D5-C2
Similarity: 0.6353
Relevance: Partially Relevant

Compare Q1 with Q3 and Q2 with Q4. These pairs express related meanings using different wording. Check their retrieved text to evaluate semantic retrieval.

OUT-OF-DOMAIN QUERY TEST


,Query,Rank,Chunk ID,Document Title,Similarity,Relevance,Reason
12,How do quantum computers implement quantum ent...,1,D3-C1,Attention Mechanism in Transformers,0.1941,Irrelevant,The corpus contains no document about quantum ...
13,How do quantum computers implement quantum ent...,2,D5-C2,Retrieval-Augmented Generation,0.1899,Irrelevant,The corpus contains no document about quantum ...
14,How do quantum computers implement quantum ent...,3,D3-C2,Attention Mechanism in Transformers,0.1821,Irrelevant,The corpus contains no document about quantum ...


FAISS returns the nearest available vectors even if the corpus does not contain the answer. Similarity alone does not prove that a retrieved passage answers the question.

EXPERIMENT 8: EXECUTION SUMMARY
Documents loaded: 5
Chunks created: 17
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
FAISS index type: IndexFlatIP
Similarity metric: Cosine similarity
Vectors indexed: 17
Test queries executed: 5
Top-k per query: 3
Total retrieved results: 15

Experiment execution completed successfully.
